In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
from typing import List, Tuple, Callable, Union, Dict, Any, Optional
from collections import defaultdict
from numpy.polynomial.legendre import leggauss
# from pyomo.environ import *
from scipy.optimize import linprog
from sympy.core.relational import Relational as SympyRelational
import itertools as itools
from sympy.logic.boolalg import BooleanTrue, BooleanFalse
import numpy as np
import chaospy as cp
import math
import pickle
from pathlib import Path
import sympy as sp
import time
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import ConcreteModel, ConstraintList, Var, RangeSet, Binary, NonNegativeReals, Param, Constraint
from tqdm.auto import tqdm   # auto-detects notebook vs terminal
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gurobipy as gp
from gurobipy import GRB
from gurobipy import nlfunc

## Smolyak quadrature

In [2]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)

    for region in solution.critical_regions:
        if region.is_inside(theta_vector.reshape(-1, 1)):
            coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
            max_coefficients = coefficients[max_idx]
            min_coefficients = coefficients[min_idx]
            # theta_max = float(max_coefficients @ theta_vector_aug)
            # theta_min = float(min_coefficients @ theta_vector_aug)
            theta_max = (max_coefficients @ theta_vector_aug).item()
            theta_min = (min_coefficients @ theta_vector_aug).item()
            return theta_min, theta_max

    raise ValueError(
        "The provided theta_vector is not inside any critical region of the solution.")


def map_u_to_theta_and_jacobian(solutions: List, u: np.ndarray, d_vector: np.ndarray) -> tuple:
    """Given the parametric solutions for theta_k and the current 'state' vector (theta_prev + d),
        return the theta_k vector and the Jacobian matrix dtheta/du.
    Args:
        solutions (List): List of parametric solutions for each theta_k.
        u (np.ndarray): 1D arraay of canonical coordinates 
        d_vector (np.ndarray): Current disturbance vector.
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if isinstance(d_vector, np.ndarray):
            theta_vector = np.block([np.array(theta_values), d_vector])
        else:
            theta_vector = np.array(theta_values, dtype=float)

        theta_min, theta_max = theta_interval_at_point(sol, theta_vector)
        length = theta_max - theta_min

        theta_k = 0.5 * length * u[k] + 0.5 * (theta_max + theta_min)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian") -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    nodes_u, weights_expectation = cp.quadrature.sparse_grid(
        order=level, dist=dist, rule=rule)

    weights_u = weights_expectation * (2.0 ** n_theta)

    nodes_u = nodes_u.T

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(nodes_u.shape[0]):
        u_vector = nodes_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
        stochastic_flexibility += func_value * jacobian * weights_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

## Gaussian Legendre quadrature

In [3]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n,) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1, 1)

    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) +
                    np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1, -1)

    return points, weights


def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped

    for region in solution.critical_regions:
        if region.is_inside(t_vector.reshape(-1, 1)):
            coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
            qpoints, qweights = gauss_legendre_between_bounds(
                expr_coeffs=coeffs, n_gl=nq)
            return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug

    # print(f't_vector: {t_vector}')
    # print(f'solution:{solution}')
    raise ValueError("No region found that contains the given t_vector.")


def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError(
                "If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )

        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector,
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
              )

    return stflex

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [5]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None, mp_algo: mpqp_algorithm = mpqp_algorithm.combinatorial):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)

    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("error", category=UserWarning, message="The chebychev ball has either a radius of zero, or the problem is not feasible!")
                    prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)

                prob.process_constraints()
                solution = solve_mpqp(problem=prob, algorithm=mp_algo)

                prob_dict[f"t{i}"] = prob
                theta_bound_dict[f"t{i}"] = solution

            except UserWarning as w:
                # MPLP infeasible / degenerate
                print(f"[theta {i}] MPLP infeasible / zero Chebyshev ball: {w}")
                prob_dict[f"t{i}"] = None
                theta_bound_dict[f"t{i}"] = None

        else:
            # LP fallback branch via scipy.optimize.linprog
            linres = linprog(c=c, A_ub=A, b_ub=b)

            if not linres.success:
                print(f"[theta {i}] linprog failed: status={linres.status}, "f"message={linres.message}")
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = None
            else:
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = [linres.x[1], linres.x[0]]

        print(f"Finished solving for theta{i+1}")

    probs = [p for _, p in prob_dict.items()]
    sols = [sol for _, sol in theta_bound_dict.items()]
    return probs, sols

In [6]:
def get_bounds_regions(sols: List, min_idx: int = 1, max_idx: int = 0):
    theta_bounds_list = []
    theta_regions_list = []

    for theta_sol in sols:
        if theta_sol is None:
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        if not getattr(theta_sol, "critical_regions", None):
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        min_max_list = []
        region_list = []

        for cr in theta_sol.critical_regions:
            # Store bounds
            Ab = np.concatenate([cr.A, cr.b], axis=1)[:2]
            min_max_list.append([Ab[min_idx].tolist(), Ab[max_idx].tolist()])

            # Store region constraints
            Ef = np.concatenate([cr.E, -cr.f], axis=1)
            region_array = np.array([row.tolist() for row in Ef], dtype=float)
            region_list.append(region_array)

        # Append per-theta data
        theta_bounds_list.append(np.array(min_max_list))
        # <-- each region is a 2D array
        theta_regions_list.append(np.array(region_list, dtype=object))

    return theta_bounds_list, theta_regions_list


def generate_region_combos(region_sizes, n_gl):
    """Generate region index combinations based on critical region structure."""
    n_theta = len(region_sizes)
    region_combo_shape = []
    for k in range(n_theta):
        n_paths = int(np.prod(n_gl[:k])) if k > 0 else 1
        region_combo_shape.extend([range(region_sizes[k])] * n_paths)
    return list(itools.product(*region_combo_shape))


def affine_expr(coeffs, symbols):
    return sum(c * s for c, s in zip(coeffs[:-1], symbols)) + coeffs[-1]


def normalized_lhs(ineq):
    return ineq.lhs.expand() if hasattr(ineq, 'lhs') else None

In [7]:
def _prepare_state_data(state, tbounds, dbounds, *, solve_algo, theta_algo, log=False):
    """
    Build and solve the flexibility problem for one state, then return only valid theta-related data.

    Returns
    -------
    dict with keys:
        state
        flex_sol
        sol_list
        theta_bounds_list
        theta_regions_list
        filtered_solutions
        filtered_theta_bounds
        filtered_theta_regions
    or None if no valid theta regions exist.
    """
    state_model = create_flexibility_model(y_list=state, tbounds=tbounds, dbounds=dbounds)
    state_prob = state_model.formulate_problem()
    state_prob.process_constraints()

    flex_sol = solve_mpqp(problem=state_prob, algorithm=solve_algo)

    if log:
        print(f"Number of critical regions in for flexibility function for state {state}: {len(flex_sol.critical_regions)}")

    _, sol_list = get_theta_bounds(flex_sol=flex_sol, numt=nt, numd=nd, tbounds=t_bounds, dbounds=dbounds, mp_algo=theta_algo)

    theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)

    filtered = [(sol, tb, tr) for sol, tb, tr in zip(sol_list, theta_bounds_list, theta_regions_list) if sol is not None and tb is not None and tr is not None]

    if not filtered:
        return None

    filtered_solutions = [sol for sol, _, _ in filtered]
    filtered_theta_bounds = [tb for _, tb, _ in filtered]
    filtered_theta_regions = [tr for _, _, tr in filtered]

    return {
        "state": state,
        "flex_sol": flex_sol,
        "sol_list": sol_list,
        "theta_bounds_list": theta_bounds_list,
        "theta_regions_list": theta_regions_list,
        "filtered_solutions": filtered_solutions,
        "filtered_theta_bounds": filtered_theta_bounds,
        "filtered_theta_regions": filtered_theta_regions,
    }

In [8]:
def _safe_sf_call(func, state, label, **kwargs):
    try:
        return func(**kwargs)
    except ValueError as e:
        if "not inside any critical region" in str(e):
            print(f"Skipping {label} SF for state {state}: {e}")
            return None
        raise

In [9]:
# def calculate_esf(y_d: dict, prepared_data_by_state: dict, d_v, n_q: int, s_level: int):
#     g_esf, s_esf = 0, 0
# 
#     for state, prob in y_d.items():
#         try:
#             data = prepared_data_by_state.get(state)
# 
#             if data is None:
#                 print(f"No valid theta regions for state {state}; skipping state")
#                 continue
# 
#             sols = data["filtered_solutions"]
# 
#             sf_idx_gaussian = _safe_sf_call(
#                 calculate_stocflexibility,
#                 state=state,
#                 label="Gaussian",
#                 sols=sols,
#                 nq=n_q,
#                 joint_func=joint_pdf,
#                 d_vector=d_v,
#             )
#             if sf_idx_gaussian is not None:
#                 g_esf += sf_idx_gaussian * prob
# 
#             sf_idx_smolyak = _safe_sf_call(
#                 calculate_stocflexibility_smolyak,
#                 state=state,
#                 label="Smolyak",
#                 solutions=sols,
#                 level=s_level,
#                 joint_func=joint_pdf,
#                 d_vector=d_v,
#             )
#             if sf_idx_smolyak is not None:
#                 s_esf += sf_idx_smolyak * prob
# 
#         except ValueError as e:
#             print(f"Skipping state {state} due to ValueError: {e}")
#             continue
# 
#         print(f"Finished for state {state}.")
# 
#     return g_esf, s_esf

In [10]:
def calculate_gl_esf(y_d: dict, prepared_data_by_state: dict, d_v, n_q: int):
    gl_esf = 0

    for state, prob in y_d.items():
        try:
            data = prepared_data_by_state.get(state)

            if data is None:
                print(f"No valid theta regions for state {state}; skipping state")
                continue

            sols = data["filtered_solutions"]

            sf_idx_gaussian = _safe_sf_call(
                calculate_stocflexibility,
                state=state,
                label="Gaussian",
                sols=sols,
                nq=n_q,
                joint_func=joint_pdf,
                d_vector=d_v,
            )
            if sf_idx_gaussian is not None:
                gl_esf += sf_idx_gaussian * prob

        except ValueError as e:
            print(f"Skipping state {state} due to ValueError: {e}")
            continue

        print(f"Finished for state {state}.")

    return gl_esf

In [11]:
def calculate_sm_esf(y_d: dict, prepared_data_by_state: dict, d_v, s_level: int):
    sm_esf = 0

    for state, prob in y_d.items():
        try:
            data = prepared_data_by_state.get(state)

            if data is None:
                print(f"No valid theta regions for state {state}; skipping state")
                continue

            sols = data["filtered_solutions"]

            sf_idx_smolyak = _safe_sf_call(
                calculate_stocflexibility_smolyak,
                state=state,
                label="Smolyak",
                solutions=sols,
                level=s_level,
                joint_func=joint_pdf,
                d_vector=d_v,
            )
            if sf_idx_smolyak is not None:
                sm_esf += sf_idx_smolyak * prob

        except ValueError as e:
            print(f"Skipping state {state} due to ValueError: {e}")
            continue

        print(f"Finished for state {state}.")

    return sm_esf

## Smolyak SF Expression

In [12]:
def compute_sf_exprs_regions_smolyak(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    FIXED VERSION:
    - For each region combo, contributions from Smolyak nodes are GATED by the region constraints
      after substituting theta(theta_prev, d, u_node).
    - This prevents double-counting across region combos (the main bug in the old version).

    Returns:
      sf_exprs:  list of sympy expressions (each is gated to its combo)
      sf_regions: list of region constraint lists (same as before)
    """
    n_theta = len(theta_syms)

    # Sparse grid nodes/weights on [-1,1]^n (Chaospy gives expectation weights)
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    nodes_u, weights_expectation = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )
    nodes_u = np.array(nodes_u).T
    weights_expectation = np.array(weights_expectation).flatten()

    # Convert expectation weights to integral weights over [-1,1]^n (volume = 2^n)
    weights_u = (2.0 ** n_theta) * weights_expectation

    # Enumerate region combos (same as your current code)
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = list(itools.product(*[range(r) for r in region_sizes]))

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:

        # 1) Build region constraints in (theta_syms, d_syms)
        combo_constraints = []
        for level_idx, region_idx in enumerate(region_combo):
            rows = theta_regions_list[level_idx][region_idx]
            for row in rows:
                t_coeffs = row[:level_idx+1]
                d_coeffs = row[level_idx+1:-1]
                const = row[-1]

                lhs = sum(c * theta_syms[i] for i, c in enumerate(t_coeffs)) + \
                    sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                ineq = lhs <= 0

                if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                    combo_constraints.append(ineq)

        # (Optional) stable order; not dedupe, but OK
        combo_constraints = sorted(combo_constraints, key=str)

        # 2) Smolyak quadrature expression with node-wise gating
        sf_sum = 0

        for u_vec, w_u in zip(nodes_u, weights_u):

            theta_vals = []
            jacobian = 1

            # Sequentially compute theta_k(u, d) for this combo
            for level_idx, region_idx in enumerate(region_combo):
                bounds = theta_bounds_list[level_idx][region_idx]
                bound_inputs = theta_vals + list(d_syms)

                # IMPORTANT: keep your existing convention here:
                # bounds[0] is t_min, bounds[1] is t_max (do NOT change since GL works for you)
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                u_k = float(u_vec[level_idx])
                t_k = 0.5 * (t_max - t_min) * u_k + 0.5 * (t_max + t_min)

                theta_vals.append(t_k)
                jacobian *= 0.5 * (t_max - t_min)

            # Substitute theta into PDF (so pdf becomes expression in d_syms)
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)

            # GATE this node's contribution by combo feasibility
            if combo_constraints:
                gated_constraints = []
                for ineq in combo_constraints:
                    ineq_sub = ineq.subs(theta_subs)
                    # After substitution this should depend only on d_syms (and constants)
                    if not isinstance(ineq_sub, (BooleanTrue, BooleanFalse)):
                        gated_constraints.append(ineq_sub)

                if gated_constraints:
                    cond = sp.And(*gated_constraints)
                    term = sp.Piecewise(
                        (w_u * jacobian * pdf_val, cond),
                        (0, True)
                    )
                else:
                    # constraints evaluated to True/False already
                    term = w_u * jacobian * pdf_val
            else:
                term = w_u * jacobian * pdf_val

            sf_sum += term

        sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(combo_constraints)

    return sf_exprs, sf_regions


def compute_sf_smolyak_symbolic_fast(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Smolyak symbolic SF (node-wise region selection) with SAFE Piecewise creation.

    Returns:
        sf_expr (sympy Expr), stats (dict)
    """
    n_theta = len(theta_syms)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    nodes_u, wE = cp.generate_quadrature(
        order=level, dist=dist, rule=rule, sparse=True, growth=growth
    )
    nodes_u = np.asarray(nodes_u, dtype=float).T
    wE = np.asarray(wE, dtype=float).ravel()
    wU = (2.0 ** n_theta) * wE

    def _region_holds(k: int, region_rows, theta_prev_exprs):
        """Return sympy Boolean condition (in d_syms only) for stage k region feasibility."""
        conds = []
        for row in region_rows:
            t_coeffs = row[:k]        # theta_0..theta_{k-1}
            d_coeffs = row[k:-1]      # d0..d_{nd-1}
            const = row[-1]

            lhs = sum(c * theta_prev_exprs[i] for i, c in enumerate(t_coeffs)) \
                + sum(c * d for c, d in zip(d_coeffs, d_syms)) \
                + const

            ineq = sp.Le(lhs, 0)
            if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                conds.append(ineq)

        return sp.And(*conds) if conds else sp.true

    sf_sum = 0

    for u_vec, w_u in zip(nodes_u, wU):
        theta_vals = []   # sympy expressions in d_syms
        jac = 1

        for k in range(n_theta):
            bound_inputs = theta_vals + list(d_syms)

            # Build non-nested Piecewise by collecting (expr, cond) pairs
            min_pairs = []
            max_pairs = []

            n_regions_k = theta_bounds_list[k].shape[0]
            for r_idx in range(n_regions_k):
                # [tmin_coeffs, tmax_coeffs] (keep your convention)
                bounds = theta_bounds_list[k][r_idx]
                region_rows = theta_regions_list[k][r_idx]

                cond = _region_holds(k, region_rows, theta_vals)
                tmin_r = affine_expr(bounds[0], bound_inputs)
                tmax_r = affine_expr(bounds[1], bound_inputs)

                min_pairs.append((tmin_r, cond))
                max_pairs.append((tmax_r, cond))

            # IMPORTANT: add a default branch to avoid Sympy as_set/ITE rewrite errors
            # Use the last expression as fallback. (Assumes regions cover the space; if not, it still prevents crashes.)
            pw_tmin = sp.Piecewise(*min_pairs, (min_pairs[-1][0], True))
            pw_tmax = sp.Piecewise(*max_pairs, (max_pairs[-1][0], True))

            u_k = float(u_vec[k])
            t_k = 0.5 * (pw_tmax - pw_tmin) * u_k + 0.5 * (pw_tmax + pw_tmin)

            theta_vals.append(t_k)
            jac *= 0.5 * (pw_tmax - pw_tmin)

        pdf_val = joint_pdf_expr.subs(
            {sym: val for sym, val in zip(theta_syms, theta_vals)})
        sf_sum += w_u * jac * pdf_val

    stats = {"n_nodes": int(nodes_u.shape[0])}
    # Avoid simplify() here; it can take forever on Piecewise-heavy expressions
    return sf_sum, stats

## Gaussian Legendre SF Expressions

In [13]:
def compute_sf_exprs_regions(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    n_gl_list,
    theta_syms
):
    n_theta = len(theta_syms)
    quad_data = [np.polynomial.legendre.leggauss(n) for n in n_gl_list]
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = generate_region_combos(region_sizes, n_gl_list)

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:
        combo_ptr = 0
        # Initialize integration paths: (theta_vals, weight, scale, constraints)
        paths = [([], 1, 1, [])]

        for level in range(n_theta):
            xi, wi = quad_data[level]
            new_paths = []

            for theta_vals, weight, scale, constraints in paths:
                region_idx = region_combo[combo_ptr]
                combo_ptr += 1

                bound_inputs = theta_vals + list(d_syms)
                bounds = theta_bounds_list[level][region_idx]
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                # Get level-specific region constraints
                rows = theta_regions_list[level][region_idx]
                level_constraints = []
                for row in rows:
                    t_coeffs = row[:level]
                    d_coeffs = row[level:-1]
                    const = row[-1]
                    lhs = sum(c * theta_vals[i] for i, c in enumerate(t_coeffs)) + \
                        sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                    ineq = lhs <= 0
                    # level_constraints.append(sp.simplify(lhs <= 0))
                    if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                        level_constraints.append(ineq)

                new_constraints = constraints + level_constraints

                # Quadrature expansion for this level
                for q in range(len(xi)):
                    t = 0.5 * (t_max - t_min) * xi[q] + 0.5 * (t_max + t_min)
                    # new_theta_vals = theta_vals + [sp.simplify(t)]
                    new_theta_vals = theta_vals + [t]
                    new_weight = weight * wi[q]
                    new_scale = scale * 0.5 * (t_max - t_min)
                    new_paths.append(
                        (new_theta_vals, new_weight, new_scale, new_constraints))

            paths = new_paths

        # Final integration and region collection
        sf_sum = 0
        all_constraints = []
        for theta_vals, weight, scale, constraints in paths:
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)
            sf_sum += weight * scale * pdf_val
            all_constraints.extend(constraints)

        # Deduplicate constraints symbolically
        unique_constraints = []
        for c in all_constraints:
            if isinstance(c, (BooleanTrue, BooleanFalse)):
                print(f'Skipping trivial constraint: {c}')
            if not any(normalized_lhs(c) == normalized_lhs(u) and type(c) == type(u) for u in unique_constraints if normalized_lhs(u) is not None):
                unique_constraints.append(c)

        sf_exprs.append(sf_sum)
        # sf_regions.append(sorted(all_constraints, key=str))
        # sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(sorted(unique_constraints, key=str))

    return sf_exprs, sf_regions

## Gurobipy Model

In [14]:
def build_base_model_esf_gurobi(d_bounds, obj_builder=None, esf_target_init=0.0):
    m = gp.Model("ESF_design")

    d = m.addVars(
        len(d_bounds),
        lb=[b[0] for b in d_bounds],
        ub=[b[1] for b in d_bounds],
        name="d"
    )

    ESF = m.addVar(lb=-GRB.INFINITY, name="ESF")

    m._esf_terms = []

    # esf_target_con = m.addConstr(ESF >= esf_target_init, name="esf_target")

    if obj_builder is None:
        obj_expr = gp.quicksum(d[j] for j in d.keys())
    else:
        obj_expr = obj_builder(d)

    m.setObjective(ESF, GRB.MAXIMIZE)
    m.Params.NonConvex = 2
    m.update()

    return m, d, ESF

In [15]:
def add_smolyak_sf_block_gurobi(
    m,
    name,
    d_vars,
    theta_bounds_list,
    theta_regions_list,
    nodes_u,
    weights_u,
    pdf_builder,
    prob,
    big_m,
    s_domain_lb=8.0 + 1e-6,
):
    """
    Add one Smolyak SF block to an existing gurobipy ESF model.

    Expected model-side setup before calling:
        m._esf_terms = []

    Parameters
    ----------
    m : gurobipy.Model
    name : str
        Prefix for variable/constraint names.
    d_vars : tupledict
        Design variables from build_base_model_esf_gurobi().
    theta_bounds_list : list
        Per-theta list of region-wise affine bounds. For stage k:
        theta_bounds_list[k][r] = [tmin_coeffs, tmax_coeffs]
    theta_regions_list : list
        Per-theta list of region-wise inequality rows.
    nodes_u : np.ndarray
        Shape (n_nodes, n_theta), canonical Smolyak nodes in [-1,1]^n.
    weights_u : np.ndarray
        Shape (n_nodes,), integral weights.
    pdf_builder : callable
        Signature: pdf_builder(theta, n, d_vars) -> gurobipy nonlinear expression
    prob : float
        Probability weight for this state.
    big_m : float
        Big-M for region activation constraints.
    s_domain_lb : float
        Lower bound to enforce theta[0,n] > 8 for the current PDF.
    """

    if not hasattr(m, "_esf_terms"):
        m._esf_terms = []

    n_theta = len(theta_bounds_list)
    n_nodes = int(nodes_u.shape[0])

    # Variables
    tmin = {}
    tmax = {}
    theta = {}
    jac = {}
    z = {}

    for n in range(n_nodes):
        jac[n] = m.addVar(lb=0.0, name=f"{name}_jac[{n}]")
        for k in range(n_theta):
            tmin[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_tmin[{k},{n}]")
            tmax[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_tmax[{k},{n}]")
            theta[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_theta[{k},{n}]")

    for k in range(n_theta):
        R_k = theta_bounds_list[k].shape[0]
        for n in range(n_nodes):
            for r in range(R_k):
                z[k, r, n] = m.addVar(vtype=GRB.BINARY, name=f"{name}_z[{k},{r},{n}]")

    m.update()

    d_list = [d_vars[j] for j in sorted(d_vars.keys())]

    # 1) Exactly one region per stage/node
    for k in range(n_theta):
        R_k = theta_bounds_list[k].shape[0]
        for n in range(n_nodes):
            m.addConstr(
                gp.quicksum(z[k, r, n] for r in range(R_k)) == 1,
                name=f"{name}_region_select[{k},{n}]"
            )

    # 2) Bound linking + region feasibility
    for k in range(n_theta):
        R_k = theta_bounds_list[k].shape[0]

        for n in range(n_nodes):
            for r in range(R_k):
                bounds = theta_bounds_list[k][r]  # [tmin_coeffs, tmax_coeffs]

                def affine_from_coeffs(coeffs):
                    expr = coeffs[-1]
                    for i in range(k):
                        expr += coeffs[i] * theta[i, n]
                    for j, dvar in enumerate(d_list):
                        expr += coeffs[k + j] * dvar
                    return expr

                tmin_aff = affine_from_coeffs(bounds[0])
                tmax_aff = affine_from_coeffs(bounds[1])

                m.addConstr(
                    tmin[k, n] >= tmin_aff - big_m * (1 - z[k, r, n]),
                    name=f"{name}_tmin_lb[{k},{r},{n}]"
                )
                m.addConstr(
                    tmin[k, n] <= tmin_aff + big_m * (1 - z[k, r, n]),
                    name=f"{name}_tmin_ub[{k},{r},{n}]"
                )
                m.addConstr(
                    tmax[k, n] >= tmax_aff - big_m * (1 - z[k, r, n]),
                    name=f"{name}_tmax_lb[{k},{r},{n}]"
                )
                m.addConstr(
                    tmax[k, n] <= tmax_aff + big_m * (1 - z[k, r, n]),
                    name=f"{name}_tmax_ub[{k},{r},{n}]"
                )

                rows = theta_regions_list[k][r]
                for row_idx, row in enumerate(rows):
                    t_coeffs = row[:k]
                    d_coeffs = row[k:-1]
                    const = row[-1]

                    lhs = const
                    for i in range(k):
                        lhs += t_coeffs[i] * theta[i, n]
                    for j, dvar in enumerate(d_list):
                        lhs += d_coeffs[j] * dvar

                    m.addConstr(
                        lhs <= big_m * (1 - z[k, r, n]),
                        name=f"{name}_region_ineq[{k},{r},{n},{row_idx}]"
                    )

    # 3) Theta mapping + Jacobian product
    for n in range(n_nodes):
        u_vec = nodes_u[n, :]
        scale_vars = []

        for k in range(n_theta):
            u_k = float(u_vec[k])

            m.addConstr(
                theta[k, n] == 0.5 * (tmax[k, n] - tmin[k, n]) * u_k
                               + 0.5 * (tmax[k, n] + tmin[k, n]),
                name=f"{name}_theta_map[{k},{n}]"
            )

            scale_k = m.addVar(lb=0.0, name=f"{name}_scale[{k},{n}]")
            m.addConstr(
                scale_k == 0.5 * (tmax[k, n] - tmin[k, n]),
                name=f"{name}_scale_def[{k},{n}]"
            )
            scale_vars.append(scale_k)

        # Jacobian = product of scale factors
        if n_theta == 1:
            m.addConstr(jac[n] == scale_vars[0], name=f"{name}_jac_def[{n}]")
        else:
            v_prev = scale_vars[0]
            for k in range(1, n_theta):
                v_cur = m.addVar(lb=0.0, name=f"{name}_jac_step[{k},{n}]")
                m.addConstr(v_cur == v_prev * scale_vars[k], name=f"{name}_jac_chain[{k},{n}]")
                v_prev = v_cur

            m.addConstr(jac[n] == v_prev, name=f"{name}_jac_def[{n}]")

        # PDF domain guard for your current pdf_builder: x = theta[0,n] - 8 > 0
        m.addConstr(theta[0, n] >= s_domain_lb, name=f"{name}_pdf_domain[{n}]")

    # 4) Block SF expression
    SF_block = 0
    for n in range(n_nodes):
        pdf_expr = pdf_builder(theta, n)
        SF_block += float(weights_u[n]) * jac[n] * pdf_expr

    # 5) Store weighted term for final ESF definition
    m._esf_terms.append(float(prob) * SF_block)

    return {
        "tmin": tmin,
        "tmax": tmax,
        "theta": theta,
        "jac": jac,
        "z": z,
        "sf_expr": SF_block,
    }

In [16]:
def smolyak_nodes_weights(
    n_theta: int,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Returns:
        nodes_u: (N, n_theta) array of Smolyak nodes in u-space (each u_k in [-1,1])
        weights_u: (N,) array of weights for integrating over [-1,1]^n_theta
                   i.e., sum_i weights_u[i] * f(nodes_u[i]) ≈ ∫_{[-1,1]^n} f(u) du
    """
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    # Chaospy returns nodes shape (n_theta, N) and weights for expectation
    nodes, wE = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )

    nodes = np.asarray(nodes, dtype=float)        # (n_theta, N)
    wE = np.asarray(wE, dtype=float).ravel()      # (N,)

    nodes_u = nodes.T                             # (N, n_theta)

    # Convert expectation weights to integral weights over [-1,1]^n:
    # E[f(U)] = ∫ f(u) p(u) du with p(u)=1/2^n on [-1,1]^n
    # => ∫ f(u) du = 2^n * E[f(U)]
    weights_u = (2.0 ** n_theta) * wE

    return nodes_u, weights_u

In [17]:
# # Bansal (2000) Illustrative Example
# y_dict = {
#     (0.5,0.5): 0.01,
#     (0.5,1): 0.09,
#     (1,0.5): 0.09,
#     (1,1): 0.81,
# }
# 
# t_bounds = [(0, 4), (0, 4)]
# d_bounds = [(0, 5), (0, 5)]
# nt = len(t_bounds)
# nd = len(d_bounds)
# 
# def create_flexibility_model(tbounds:list, dbounds:list, y_list: tuple = None):
#     m = MPModeler()
# 
#     u = m.add_var(name='u')
#     x = m.add_var(name='x')
#     z = m.add_var(name='z')
# 
#     t1 = m.add_param(name='t1')
#     t2 = m.add_param(name='t2')
#     d1 = m.add_param(name='d1')
#     d2 = m.add_param(name='d2')
#     m.add_constr(2*x - 3*z + t1 - d2 == 0)
#     m.add_constr(x - z/2 - t1/2 + t2/2 + d1 - 7*d2/2 <= u)
#     m.add_constr(-2*x + 2*z - 4*t1/3 - t2 + 2*d2 + 1/3 <= u)
#     m.add_constr(-x + 5*z/2 + t1/2 - t2 - d1 + d2/2 - 1 <= u)
#     m.add_constr(-50 <= x)
#     m.add_constr(-50 <= z)
#     m.add_constr(tbounds[0][0] <= t1)
#     m.add_constr(tbounds[1][0] <= t2)
#     m.add_constr(dbounds[0][0] <= d1)
#     m.add_constr(dbounds[1][0] <= d2)
#     m.add_constr(t1 <= tbounds[0][1])
#     m.add_constr(t2 <= tbounds[1][1])
#     m.add_constr(d1 <= dbounds[0][1] * y_list[0])
#     m.add_constr(d2 <= dbounds[1][1] * y_list[1])
#     m.set_objective(u)
# 
#     return m
# 
# def joint_pdf(theta: list):
#     return (2/np.pi)*np.exp(-2*((theta[0]-2)**2 + (theta[1]-2)**2))
# 
# def pdf_expr(block, n):
#     theta0 = block.theta[0, n]
#     theta1 = block.theta[1, n]
#     return (2/math.pi) * pyo.exp(-2*((theta0-2)**2 + (theta1-2)**2))

In [18]:
# Bansal (2000) Process Example 1
y_dict = {
    (0, 0, 0): 0.001,
    (0, 0, 1): 0.003,
    (0, 1, 0): 0.006,
    (1, 0, 0): 0.010,
    (0, 1, 1): 0.040,
    (1, 0, 1): 0.066,
    (1, 1, 0): 0.114,
    (1, 1, 1): 0.760
}

# y_dict = {(1,1,1):1}

t_bounds = [(8, 16), (3, 11)]
d_bounds = [(0, 10), (0, 10), (0, 10)]
nt = len(t_bounds)
nd = len(d_bounds)


def create_flexibility_model(tbounds: list, dbounds: list, y_list: tuple = None):
    j1 = 0.92
    j2 = 0.85
    j3 = 0.75

    m = MPModeler()

    u = m.add_var(name='u')
    F1 = m.add_var(name="F1")
    F2 = m.add_var(name="F2")
    F3 = m.add_var(name="F3")
    F4 = m.add_var(name="F4")
    F5 = m.add_var(name="F5")
    F6 = m.add_var(name="F6")
    F7 = m.add_var(name="F7")

    S = m.add_param(name='S')
    D = m.add_param(name='D')

    d1 = m.add_param(name='d1')
    d2 = m.add_param(name='d2')
    d3 = m.add_param(name='d3')

    m.add_constr(F4 - j1 * F2 == 0)
    m.add_constr(F1 - F2 - F3 == 0)
    m.add_constr(F5 - j2 * F4 == 0)
    m.add_constr(F6 - j3 * F3 == 0)
    m.add_constr(F7 - F5 - F6 == 0)
    m.add_constr(F1 - S <= u)
    m.add_constr(D - F7 <= u)
    m.add_constr(F2 - d1 * y_list[0] <= u)
    m.add_constr(F4 - d2 * y_list[1] <= u)
    m.add_constr(F3 - d3 * y_list[2] <= u)

    # for v in [F1, F2, F3, F4, F5, F6, F7]:
    #     m.add_constr(v >= 0)

    m.add_constr(tbounds[0][0] + 1e-6 <= S)
    m.add_constr(S <= tbounds[0][1])
    m.add_constr(tbounds[1][0] + 1e-6 <= D)
    m.add_constr(D <= tbounds[1][1])

    m.add_constr(dbounds[0][0] <= d1)
    m.add_constr(d1 <= dbounds[0][1])
    m.add_constr(dbounds[1][0] <= d2)
    m.add_constr(d2 <= dbounds[1][1])
    m.add_constr(dbounds[2][0] <= d3)
    m.add_constr(d3 <= dbounds[2][1])

    m.set_objective(u)

    return m

def joint_pdf(theta: list):
    Sval, Dval = theta
    eps = 1e-12
    # x = max(Sval - 8.0, eps)
    return (1 / (1.2 * np.pi * (Sval - 8.0))) * np.exp(
        -1.39 * (np.log(Sval - 8.0)) ** 2 - 0.5 * (Dval - 7.0) ** 2)


def pdf_builder(theta, n):
    eps = 1e-6
    S = theta[0, n]
    D = theta[1, n]
    x = S - 8.0

    return (1.0 / (1.2 * math.pi)) * (1.0 / (x + eps)) * nlfunc.exp(
        -1.39 * (nlfunc.log(x + eps) * nlfunc.log(x + eps))
        - 0.5 * ((D - 7.0) * (D - 7.0))
    )

In [19]:
prepared_data_by_state = {
    state: _prepare_state_data(
        state,
        tbounds=t_bounds,
        dbounds=d_bounds,
        solve_algo=mpqp_algorithm.combinatorial,
        theta_algo=mpqp_algorithm.combinatorial,
    )
    for state in y_dict
}

Set parameter Username
Set parameter LicenseID to value 2798285
Academic license - for non-commercial use only - expires 2027-03-25
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
Finished solving for theta1
Finished solving for theta2
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible /

In [20]:
nodes_u, weights_u = smolyak_nodes_weights(n_theta=len(t_bounds), level=4, rule="gaussian", growth=True)

m_sf_sm_exact, d_vars, ESF = build_base_model_esf_gurobi(d_bounds=d_bounds)

Set parameter NonConvex to value 2


In [21]:
for state, prob in y_dict.items():
    data = prepared_data_by_state[state]
    if data is None:
        continue

    theta_bounds_list = data["filtered_theta_bounds"]
    theta_regions_list = data["filtered_theta_regions"]

    add_smolyak_sf_block_gurobi(
        m=m_sf_sm_exact,
        name=f"sf_{state}",
        d_vars=d_vars,
        theta_bounds_list=theta_bounds_list,
        theta_regions_list=theta_regions_list,
        nodes_u=nodes_u,
        weights_u=weights_u,
        pdf_builder=pdf_builder,
        prob=prob,
        big_m=1e4,
    )

Warning for adding constraints: zero or small (< 1e-13) coefficients, ignored


In [22]:
esf_def_con = m_sf_sm_exact.addConstr(
    ESF == gp.quicksum(m_sf_sm_exact._esf_terms),
    name="esf_def"
)
m_sf_sm_exact.update()

print(esf_def_con)

<gurobi.GenConstr 0>


In [ ]:
print(m_sf_sm_exact.getConstrByName("esf_def"))

In [ ]:
m_sf_sm_exact.update()

In [ ]:
m_sf_sm_exact.Params.FuncNonlinear = 1  # optional, depending on how you wrote pdf_builder[web:316]

In [ ]:
# esf_target_con.RHS = 0.3
m_sf_sm_exact.Params.MIPGap = 0.05
m_sf_sm_exact.update()

In [ ]:
m_sf_sm_exact.optimize()

In [ ]:
d_vars[0].X

In [ ]:
d_vars[1].X

In [ ]:
d_vars[2].X

In [ ]:
ESF.X

In [ ]:
m_sf_sm_exact.ObjVal

In [ ]:
design_vector = np.array([d_vars[j].X for j in range(nd)])

In [ ]:
design_vector

In [ ]:
esf_gl = calculate_gl_esf(y_d=y_dict, prepared_data_by_state=prepared_data_by_state, d_v=design_vector, n_q=16)

In [ ]:
print(f'Gauss Legendre ESF: {esf_gl}')

In [ ]:
esf_sm = calculate_sm_esf(y_d=y_dict, prepared_data_by_state=prepared_data_by_state, d_v=design_vector, s_level=16)

In [ ]:
print(f'Smolyak ESF: {esf_sm}')

In [ ]:
esf_ub = ESF.X
esf_lb = esf_sm

In [ ]:
rel_error = abs(esf_ub-esf_lb)/esf_ub *100

In [ ]:
rel_error